In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

#load dataset
orders_df = pd.read_csv('orders.csv')
items_df = pd.read_csv('order_items.csv')

#merge them 
df = pd.merge(items_df, orders_df, on='order_id')

print("Merged Data Preview:")
df[['order_id', 'cafeteria_id', 'item_id', 'quantity']].head(10)

Merged Data Preview:


,order_id,cafeteria_id,item_id,quantity
0,1,1,7,1
1,1,1,16,1
2,1,1,17,1
3,2,3,49,1
4,2,3,52,1
5,2,3,53,1
6,3,2,19,1
7,4,2,26,1
8,4,2,34,1
9,4,2,35,1


In [2]:
import json
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

#generate combo rules (apriori) 
combo_rules = {}
failing_items = {}
bogo_rules = {}

for canteen_id in df['cafeteria_id'].unique():
    canteen_data = df[df['cafeteria_id'] == canteen_id]
    cid_str = str(int(canteen_id))
    
    #combo logic
    basket = (canteen_data.groupby(['order_id', 'item_id'])['quantity']
              .sum().unstack().reset_index().fillna(0)
              .set_index('order_id'))
    
    basket = basket.map(lambda x: 1 if x > 0 else 0)
    frequent_itemsets = apriori(basket, min_support=0.03, use_colnames=True)
    canteen_combos = []
    
    if not frequent_itemsets.empty:
        rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
        rules = rules[(rules['antecedents'].apply(len) == 1) & (rules['consequents'].apply(len) == 1)]
        
        #sort by confidence to get the strongest rules first
        rules = rules.sort_values('confidence', ascending=False)
        
        seen_pairs = set()
        for _, r in rules.iterrows():
            item_a = int(list(r['antecedents'])[0])
            item_b = int(list(r['consequents'])[0])
            
            #prevent mirrors (A->B and B->A) by sorting the tuple
            pair = tuple(sorted([item_a, item_b]))
            if pair not in seen_pairs:
                seen_pairs.add(pair)
                canteen_combos.append({
                    "item_a": item_a,
                    "item_b": item_b,
                    "confidence": round(r['confidence'], 3), 
                    "lift": round(r['lift'], 3)
                })
                if len(canteen_combos) >= 10: break 
                    
    combo_rules[cid_str] = canteen_combos

    #failing items logic
    item_sales = canteen_data.groupby('item_id')['quantity'].sum().reset_index()
    #sort from worst selling to best selling
    item_sales = item_sales.sort_values('quantity', ascending=True) 
    
    low_sales_threshold = item_sales['quantity'].quantile(0.25)
    critical_sales_threshold = item_sales['quantity'].quantile(0.10)
    high_sales_threshold = item_sales['quantity'].quantile(0.90)
    
    failing = item_sales[item_sales['quantity'] <= low_sales_threshold]
    max_fail_qty = failing['quantity'].max() if not failing.empty else 1
    failing_list = []
    
    for _, row in failing.iterrows():
        #severity score: 1.0 is absolute worst, nearing 0.0 is closer to threshold
        severity = 1.0 - (row['quantity'] / max_fail_qty) if max_fail_qty > 0 else 1.0
        failing_list.append({
            "item_id": int(row['item_id']),
            "quantity": int(row['quantity']),
            "severity": round(severity, 3) 
        })
        
    failing_items[cid_str] = failing_list
    
    #bogo logic
    critical_items = item_sales[item_sales['quantity'] <= critical_sales_threshold]['item_id'].tolist()
    popular_items = item_sales[item_sales['quantity'] >= high_sales_threshold]['item_id'].tolist()
    
    canteen_bogos = []
    
    #type 1: clearance (buy X get X free for critically failing items)
    for item in critical_items[:3]: 
        canteen_bogos.append({
            "bogo_type": "clearance",
            "buy_item": int(item),
            "get_item": int(item)
        })
        
    #type 2: cross-sell (buy popular item A, get failing item B free)
    if popular_items and critical_items:
        #pair top popular items with remaining critical items
        for i in range(min(len(popular_items), len(critical_items), 3)):
            canteen_bogos.append({
                "bogo_type": "cross_sell",
                "buy_item": int(popular_items[-i-1]), 
                "get_item": int(critical_items[i])   
            })
            
    bogo_rules[cid_str] = canteen_bogos

#save the final json files
with open('combo_rules.json', 'w') as f: json.dump(combo_rules, f, indent=4)
with open('failing_items.json', 'w') as f: json.dump(failing_items, f, indent=4)
with open('bogo_rules.json', 'w') as f: json.dump(bogo_rules, f, indent=4)

print("Dynamic discount strategies generated successfully!")

Dynamic discount strategies generated successfully!


d:\2nd Year\2nd Sem\Group Project\Demeter-Clone\ai-service\venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
d:\2nd Year\2nd Sem\Group Project\Demeter-Clone\ai-service\venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
d:\2nd Year\2nd Sem\Group Project\Demeter-Clone\ai-service\venv\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
